NAME : ADEWOLE TOLUWALOPE DAVID

In [4]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [7]:
customer_spending_raw = pd.read_csv('customer_spending.csv')
customer_spending = customer_spending_raw.copy()

In [8]:
customer_spending.info()

<class 'pandas.DataFrame'>
RangeIndex: 34866 entries, 0 to 34865
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sale_date     34866 non-null  str    
 1   sale_year     34866 non-null  int64  
 2   sale_month    34866 non-null  str    
 3   age           34866 non-null  int64  
 4   gender        34866 non-null  str    
 5   country       34866 non-null  str    
 6   state         34866 non-null  str    
 7   category      34866 non-null  str    
 8   sub_category  34866 non-null  str    
 9   quantity      34866 non-null  int64  
 10  unit_cost     34866 non-null  float64
 11  unit_price    34866 non-null  float64
 12  cost          34866 non-null  int64  
 13  revenue       34866 non-null  int64  
dtypes: float64(2), int64(5), str(7)
memory usage: 3.7 MB


Write a query that returns each category and the corresponding total revenue for that 
category for the sale_year 2016. The aggregated column should be named 
total_revenue. The output should be arranged alphabetically. 

In [9]:
result = (
    customer_spending[customer_spending['sale_year'] == 2016]
    .groupby('category', as_index=False)['revenue']
    .sum()
    .rename(columns={'revenue': 'total_revenue'})
    .sort_values('category')
)

result

,category,total_revenue
0,Accessories,4594897
1,Bikes,5722257
2,Clothing,2079651


Write a query that returns a list of sub_categories and their corresponding average 
unit_price (named avg_unit_price), average unit_cost named avg_unit_cost, as well as 
the difference between these two values (named margin) for the sale_year 2015. Round 
all values to two decimal places. Organize the results alphabetically.  

In [14]:
result = (
    customer_spending[customer_spending['sale_year'] == 2015]
    .groupby('sub_category', as_index=False)
    .agg(
        avg_unit_price=('unit_price', 'mean'),
        avg_unit_cost=('unit_cost', 'mean')
    )
    .assign(
        margin=lambda data: data['avg_unit_price'] - data['avg_unit_cost']
    )
    .round(2)
    .sort_values('sub_category')
)

result

,sub_category,avg_unit_price,avg_unit_cost,margin
0,Bike Stands,602.00,528.61,73.40
1,Bottles and Cages,75.82,66.88,8.94
2,Caps,99.96,90.38,9.58
3,Cleaners,94.85,84.20,10.65
4,Helmets,368.32,322.19,46.13
5,Hydration Packs,545.69,481.44,64.25
6,Jerseys,511.79,462.61,49.18
7,Mountain Bikes,1108.03,1147.46,-39.43
8,Road Bikes,799.99,818.16,-18.17
9,Shorts,728.04,679.28,48.76


Write a query that returns the sale_year and corresponding total number of female 
buyers (gender) for each sale_year who made purchases in the Clothing category. 
Name the aggregated column total_female_buyers. 

In [15]:
result = (
    customer_spending[
        (customer_spending['category'] == 'Clothing') &
        (customer_spending['gender'] == 'F')
    ]
    .groupby('sale_year', as_index=False)
    .agg(total_female_buyers=('gender', 'count'))
)

result

,sale_year,total_female_buyers
0,2015,1037
1,2016,1468


 Write a query that returns the age, sub_cateogry, average quantity (as a whole 
number, named avg_quantity), and average cost of products (rounded to 2 decimals, 
named avg_cost) purchased by each age and sub_category. Organize the data by age, 
oldest to youngest, and then by sub_category alphabetically. 

In [18]:
result = (
    customer_spending
    .groupby(['age', 'sub_category'], as_index=False)
    .agg(
        avg_quantity=('quantity', 'mean'),
        avg_cost=('cost', 'mean')
    )
    .assign(
        avg_quantity=lambda data: data['avg_quantity'].astype(int),
        avg_cost=lambda data: data['avg_cost'].round(2)
    )
    .sort_values(
        ['age', 'sub_category'],
        ascending=[False, True]
    )
)

result

,age,sub_category,avg_quantity,avg_cost
889,87,Bike Racks,3,240.00
890,87,Tires and Tubes,2,44.50
886,86,Fenders,3,637.00
887,86,Tires and Tubes,2,166.50
888,86,Vests,1,1842.00
...,...,...,...,...
11,17,Shorts,2,787.50
12,17,Socks,2,171.00
13,17,Tires and Tubes,2,204.34
14,17,Touring Bikes,2,1278.71


Write a query that returns a list of countries where more than 900 transactions were 
made by customers between the ages of 18-25 (inclusive).  

In [20]:
result = (
    customer_spending[
        customer_spending['age'].between(18, 25, inclusive='both')
    ]
    .groupby('country')
    .size()
    .loc[lambda counts: counts > 900]
    .reset_index(name='total_transactions')
    [['country']]
)

result

,country
0,Germany
1,United Kingdom
2,United States


Write a query to identify which sale_year and category combinations were the most 
profitable. Return the columns sale_year, category, the total revenue as total_revenue, 
the total cost as total_cost, and the difference between the two as profit for each 
sale_year and category. Include only rows that were profitable and sort the results from 
highest profit to lowest.

In [23]:
result = (
    customer_spending
    .groupby(['sale_year', 'category'], as_index=False)
    .agg(
        total_revenue=('revenue', 'sum'),
        total_cost=('cost', 'sum')
    )
    .assign(
        profit=lambda data: data['total_revenue'] - data['total_cost']
    )
    .query('profit > 0')
    .sort_values('profit', ascending=False)
)

result

,sale_year,category,total_revenue,total_cost,profit
3,2016,Accessories,4594897,3559646,1035251
4,2016,Bikes,5722257,5209322,512935
5,2016,Clothing,2079651,1654855,424796
0,2015,Accessories,2825767,2482249,343518
2,2015,Clothing,1357906,1237470,120436


Write a query that calculates the average spending per age for male customers 
(gender), where spending is based on the product of unit_price and quantity. Return 
the age and the calculated average spending, rounded to two decimal places, as 
avg_spending. Organize the results from highest to lowest avg_spending. 

In [26]:
result = (
    customer_spending[customer_spending['gender'] == 'M']
    .assign(spending=lambda data: data['unit_price'] * data['quantity'])
    .groupby('age', as_index=False)
    .agg(avg_spending=('spending', 'mean'))
    .assign(avg_spending=lambda data: data['avg_spending'].round(2))
    .sort_values('avg_spending', ascending=False)
)

result

,age,avg_spending
65,86,2026.00
56,73,1170.00
57,74,971.00
58,75,869.00
55,72,764.00
...,...,...
60,78,257.75
61,79,182.60
59,76,159.00
62,81,93.00


Write a query to determine the highest unit_cost, lowest unit_cost, and average 
unit_cost for each gender in each category. The output columns should be gender, 
category, high_cost, low_cost, avg_cost, in that order. Organize the results by gender 
and then category.  

In [27]:
result = (
    customer_spending
    .groupby(['gender', 'category'], as_index=False)
    .agg(
        high_cost=('unit_cost', 'max'),
        low_cost=('unit_cost', 'min'),
        avg_cost=('unit_cost', 'mean')
    )
    [['gender', 'category', 'high_cost', 'low_cost', 'avg_cost']]
    .sort_values(['gender', 'category'])
)

result

,gender,category,high_cost,low_cost,avg_cost
0,F,Accessories,3120.0,0.67,159.301847
1,F,Bikes,2443.0,180.00,956.653830
2,F,Clothing,2100.0,3.00,334.904970
3,M,Accessories,3240.0,0.67,166.962106
4,M,Bikes,2443.0,180.00,949.309243
5,M,Clothing,2100.0,3.00,337.600596


Write a query to determine the age distribution of customers by category and country 
for the sale_year 2016. For each category and country, calculate the age of the 
youngest and oldest customers, and the average age of customers rounded to one 
decimal place. The output columns should be category, country, youngest_customer, 
oldest_customer, and avg_customer_age, in that order. Organize your results by 
category and then by the average age of customers. 

In [28]:
result = (
    customer_spending[customer_spending['sale_year'] == 2016]
    .groupby(['category', 'country'], as_index=False)
    .agg(
        youngest_customer=('age', 'min'),
        oldest_customer=('age', 'max'),
        avg_customer_age=('age', 'mean')
    )
    .assign(
        avg_customer_age=lambda data: data['avg_customer_age'].round(1)
    )
    [['category', 'country', 'youngest_customer', 'oldest_customer', 'avg_customer_age']]
    .sort_values(['category', 'avg_customer_age'])
)

result

,category,country,youngest_customer,oldest_customer,avg_customer_age
1,Accessories,Germany,17,87,35.4
0,Accessories,France,17,84,35.6
2,Accessories,United Kingdom,17,85,36.1
3,Accessories,United States,17,78,37.8
4,Bikes,France,17,59,34.0
5,Bikes,Germany,17,72,34.6
6,Bikes,United Kingdom,17,75,34.8
7,Bikes,United States,17,72,39.7
9,Clothing,Germany,17,86,34.2
8,Clothing,France,17,84,35.3


 Write a query to return the country that has the highest average revenue (rounded to 2 
decimals). Your output columns should be country and high_sales. 

In [29]:
result = (
    customer_spending
    .groupby('country', as_index=False)
    .agg(high_sales=('revenue', 'mean'))
    .assign(high_sales=lambda data: data['high_sales'].round(2))
    .sort_values('high_sales', ascending=False)
    .head(1)
    [['country', 'high_sales']]
)

result

,country,high_sales
1,Germany,816.09
